# Multi-Representational Indexing

Multi-Representational Indexing is an advanced RAG technique where a document is represented in **multiple forms** before indexing.

For example:

```text
Document
   ↓
 ┌───────────┬─────────┬───────────┐
 Original   Summary   Keywords
   ↓          ↓          ↓
        Embeddings
            ↓
       Vector Store
            ↓
        Retrieval
```

### Why?

It improves retrieval because a user's query can match different representations of the same document.

**Basic indexing:**
`Document → Embedding → Vector Store`

**Multi-representational indexing:**
`Document → Multiple Representations → Embeddings → Vector Store`


In [ ]:
# **previous code**
# from langchain_community.document_loaders import WebBaseLoader
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# # Pahilo webpage bata document load gareko
# loader = WebBaseLoader(
#     "https://lilianweng.github.io/posts/2023-06-23-agent/"
# )
# docs = loader.load()

# # Arko webpage bata document load gareko
# loader = WebBaseLoader(
#     "https://lilianweng.github.io/posts/2024-02-05-human-data-quality/"
# )

# # Arko document lai existing docs list ma add gareko
# docs.extend(loader.load())

# # Document lai sano sano chunks ma divide gareko
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=200
# )

# docs = text_splitter.split_documents(docs)

# print(f"Total chunks: {len(docs)}")

## Smaller Dataset for Testing

Pahila hami le 2 ota webpage use gareko thiyau, jasle **109 chunks** banayeko thiyo.

Local **Llama 3** le harek chunk ko summary generate garnu parne bhayeko le execution dherai slow bhayo.

Aile testing ko lagi:

- Euta webpage matra use gareko
- `chunk_size` thulo banayeko
- Fewer chunks use gareko

### Kina?

- Summary generation chito huncha
- Local CPU/GPU ma kam load parcha
- Pipeline test garna easy huncha
- 100+ summaries generate garnu pardaina

Pipeline properly work bhayepachi hami feri **large dataset** use garna sakchhau.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Euta webpage load gareko
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")

docs = loader.load()

# Document lai chunks ma divide gareko
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)

docs = text_splitter.split_documents(docs)

print(f"Total chunks: {len(docs)}")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# Document ko summary generate garne chain
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Summarize the following document briefly:\n\n{doc}"
    )
    | ChatOllama(model="llama3:latest", temperature=0)
    | StrOutputParser()
)

# 36 oota document madhey Testing ko lagi 5 chunks matra use gareko
test_docs = docs[:5]

# Summary generate gareko
summaries = chain.batch(test_docs, config={"max_concurrency": 1})

print(f"Total summaries: {len(summaries)}")

In [ ]:
import uuid
from langchain_core.stores import InMemoryByteStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.documents import Document

# Local embedding model load gareko
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Summary haru lai vector database ma store garne
vectorstore = Chroma(collection_name="summaries", embedding_function=embeddings)

# Original documents store garne storage
store = InMemoryByteStore()

# Summary ra original document lai link garna ID use gareko
id_key = "doc_id"

# Multi-vector retriever create gareko
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

# Pratyek document ko unique ID banaeko
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Summary lai document ID sanga link gareko
summary_docs = [
    Document(page_content=summary, metadata={id_key: doc_ids[i]})
    for i, summary in enumerate(summaries)
]

# Summary embeddings vector store ma add gareko
retriever.vectorstore.add_documents(summary_docs)

# Original documents lai ID sanga store gareko
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
# Query gareko
query = "Memory in agents"

# Summary vector store bata sabai bhanda relevant summary khojeko
sub_docs = vectorstore.similarity_search(query, k=1)

# Relevant summary herne
sub_docs[0]

In [ ]:
# Query bata original document retrieve gareko
retrieved_docs = retriever.invoke(query)

# Pahilo relevant document ko first 500 characters herne
print(retrieved_docs[0].page_content[:500])